In [8]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from pathlib import Path

import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import torch

In [2]:
def load_lean_files(directory_path):
    print(f"Recursively scanning '{directory_path}' for Lean files...")
    docs = []
    # rglob handles deeply nested directories automatically
    for path in Path(directory_path).rglob("*.lean"):
        try:
            # Explicit utf-8 prevents crashes on math unicode symbols like ∀, ∃, ⊢
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
                docs.append(Document(page_content=text, metadata={"source": str(path)}))
        except Exception as e:
            print(f"Skipping {path} due to error: {e}")
    print(f"Successfully loaded {len(docs)} Lean documents.")
    return docs

docs = load_lean_files("../mathlib4/Mathlib")

Recursively scanning '../mathlib4/Mathlib' for Lean files...
Successfully loaded 8126 Lean documents.


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        r"\nnamespace\s+",
        r"\nsection\s+",
        r"\ntheorem\s+",
        r"\nlemma\s+",
        r"\ndef\s+",
        r"\ninductive\s+",
        r"\nstructure\s+",
        r"\nexample\s+",
        r"\nopen\s+",      # Sometimes splitting at open statements is clean
        r"\n\n", 
        r"\n", 
        r" "
    ],
    is_separator_regex=True,
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(docs)

In [4]:
chunks[1052]

Document(metadata={'source': '../mathlib4/Mathlib/AlgebraicGeometry/Over.lean'}, page_content='lemma Hom.isOver_iff [X.Over S] [Y.Over S] {f : X ⟶ Y} : f.IsOver S ↔ f ≫ Y ↘ S = X ↘ S :=\n  ⟨fun H ↦ H.1, fun h ↦ ⟨h⟩⟩\n\n/-! Also note the existence of `CategoryTheory.IsOverTower X Y S`. -/\n\n/-- Given `X.Over S`, this is the bundled object of `Over S`. -/\nabbrev asOver (X S : Scheme.{u}) [X.Over S] := OverClass.asOver X S\n\n/-- Given a morphism `X ⟶ Y` with `f.IsOver S`, this is the bundled morphism in `Over S`. -/\nabbrev Hom.asOver (f : X.Hom Y) (S : Scheme.{u}) [X.Over S] [Y.Over S] [f.IsOver S] :=\n  OverClass.asOverHom S f\n\nend AlgebraicGeometry.Scheme')

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5364.16it/s]


In [6]:
chunk_texts = [chunk.page_content for chunk in chunks]
raw_embeddings = embeddings.embed_documents(chunk_texts)

In [7]:
embedding_matrix = np.array(raw_embeddings).astype('float32')
dimension = embedding_matrix.shape[1]

In [ ]:
np.save('embedding_matrix.npy', embedding_matrix)
# embedding_matrix = np.load('embedding_matrix.npy')

In [11]:
import accelerate

In [12]:
model_id = "google/gemma-4-E2B-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)
pipe = pipeline(
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=300, 
    temperature=0.1
)
llm = HuggingFacePipeline(pipeline=pipe)

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [33]:
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embedding_matrix)